# Mamba 与状态空间模型（SSM）从零搭建

本 Notebook 从最基础的 SSM 出发，逐步搭建 Mamba 模型。

## 目录
1. [状态空间模型（SSM）基础](#1-状态空间模型基础)
2. [离散化与零阶保持（ZOH）](#2-离散化)
3. [结构化状态空间（S4 核心）](#3-s4)
4. [选择性状态空间（Mamba 核心）](#4-mamba)
5. [Mamba Block 完整实现](#5-mamba-block)
6. [实战：Mamba 文本生成模型](#6-实战)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np
from einops import rearrange

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('提示: 如果没有安装 einops，请运行: pip install einops')

---
## 1. 状态空间模型（SSM）基础

SSM 描述了一个隐状态 $h(t)$ 随时间连续演化的系统：

$$h'(t) = Ah(t) + Bx(t)$$
$$y(t) = Ch(t) + Dx(t)$$

- $A$ (状态矩阵): 控制隐状态的演化
- $B$ (输入矩阵): 控制输入如何影响隐状态
- $C$ (输出矩阵): 控制隐状态如何映射到输出
- $D$ (直通矩阵): 输入到输出的直接连接

可以理解为连续版本的 RNN。

In [ ]:
class ContinuousSSM:
    """连续状态空间模型（仅用于演示概念）"""
    def __init__(self, A, B, C, D):
        self.A = A  # [N, N]
        self.B = B  # [N, 1]
        self.C = C  # [1, N]
        self.D = D  # scalar

    def simulate(self, x, dt=0.01):
        """使用欧拉法模拟连续 SSM"""
        N = self.A.shape[0]
        h = np.zeros(N)
        outputs = []
        for t in range(len(x)):
            h = h + dt * (self.A @ h + self.B.flatten() * x[t])
            y = self.C.flatten() @ h + self.D * x[t]
            outputs.append(y)
        return np.array(outputs)


# 用一个简单的例子感受 SSM
N = 4  # 隐状态维度
np.random.seed(42)

# A 矩阵初始化为 HiPPO 矩阵（近似）
A = np.diag(np.linspace(-1, -4, N))  # 负对角线保证稳定性
B = np.random.randn(N, 1) * 0.5
C = np.random.randn(1, N) * 0.5
D = np.array([0.0])

ssm = ContinuousSSM(A, B, C, D)

# 输入信号
t = np.linspace(0, 2 * np.pi, 200)
x_input = np.sin(t) + 0.5 * np.sin(3 * t)

output = ssm.simulate(x_input, dt=t[1] - t[0])

plt.figure(figsize=(10, 3))
plt.plot(t, x_input, label='Input x(t)')
plt.plot(t, output, label='Output y(t)')
plt.legend()
plt.title('Continuous SSM: Input vs Output')
plt.xlabel('Time')
plt.show()
print('SSM 对输入信号进行了滤波/变换，可以看作一种信号处理系统')

---
## 2. 离散化与零阶保持（ZOH）

连续 SSM 需要离散化才能处理离散序列数据。

使用零阶保持（Zero-Order Hold）离散化：

$$\bar{A} = \exp(\Delta A)$$
$$\bar{B} = (\Delta A)^{-1}(\exp(\Delta A) - I) \cdot \Delta B$$

简化近似（当 $\Delta$ 足够小时）：
$$\bar{A} \approx (I - \Delta A)^{-1}$$
$$\bar{B} \approx (I - \Delta A)^{-1} \Delta B$$

In [ ]:
def discretize_zoh(A, B, C, D, delta):
    """零阶保持离散化
    
    Args:
        A, B, C, D: 连续 SSM 参数
        delta: 步长（可以标量或向量）
    """
    # 简化近似: A_bar = (I - delta * A)^{-1}
    N = A.shape[-1]
    I = torch.eye(N, device=A.device)

    if A.dim() == 1:
        # 对角 A (S4/Mamba 风格)
        A_bar = torch.exp(delta * A)  # 精确公式
        B_bar = (A_bar - 1) * B / A   # (e^(dA) - 1) / A * B
    else:
        A_bar = torch.linalg.inv(I - delta * A) @ A
        B_bar = torch.linalg.inv(I - delta * A) @ (delta * B)

    return A_bar, B_bar, C, D


# 离散化后的递推形式: h_t = A_bar * h_{t-1} + B_bar * x_t
# 这就是一个 RNN！
print("""离散化后的递推公式:
  h_t = A_bar * h_{t-1} + B_bar * x_t
  y_t = C * h_t + D * x_t

这与 RNN 结构完全一致:
  隐状态 h 不断更新，输出 y 由 h 决定。

但 SSM 可以通过**卷积模式**高效并行训练（见下一节）！""")

---
## 3. 结构化状态空间（S4 核心）

S4 的关键创新：SSM 可以等价地表示为**卷积**操作。

展开递推公式：
$$y_t = C \bar{A}^t \bar{B} x_0 + C \bar{A}^{t-1} \bar{B} x_1 + \cdots + C \bar{B} x_t$$

即 $y = K * x$，其中卷积核 $K = (C\bar{B}, C\bar{A}\bar{B}, C\bar{A}^2\bar{B}, \cdots)$

这意味着**训练时可以用 FFT 并行计算卷积，推理时可以用递推模式**。

In [ ]:
class S4Kernel(nn.Module):
    """S4 核心: 生成卷积核"""
    def __init__(self, d_model, d_state=16, dt_min=0.001, dt_max=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # 初始化 SSM 参数
        # A 使用 HiPPO 初始化的近似
        self.log_A_real = nn.Parameter(
            torch.log(0.5 * torch.ones(d_model, d_state))
        )
        self.A_imag = nn.Parameter(
            torch.randn(d_model, d_state) * 0.5
        )

        # B, C
        self.B = nn.Parameter(torch.randn(d_model, d_state) * 0.5)
        self.C = nn.Parameter(torch.randn(d_model, d_state) * 0.5)
        self.D = nn.Parameter(torch.ones(d_model))

        # dt (步长)
        log_dt = torch.rand(d_model) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        self.log_dt = nn.Parameter(log_dt)

    def forward(self, L):
        """生成长度为 L 的卷积核
        
        Returns:
            K: [d_model, L] 卷积核
        """
        dt = self.log_dt.exp()  # [d_model]
        A = -self.log_A_real.exp() + 1j * self.A_imag  # [d_model, d_state] 复数
        B = self.B.to(torch.complex64)  # [d_model, d_state]
        C = self.C.to(torch.complex64)  # [d_model, d_state]

        # 离散化: A_bar = exp(delta * A)
        dA = torch.exp(dt.unsqueeze(-1) * A)  # [d_model, d_state]
        dB = dt.unsqueeze(-1) * B * (dA - 1) / A  # [d_model, d_state]

        # 生成卷积核: K_t = C * A_bar^t * B_bar
        # 利用 Vandermonde 结构高效计算
        dA_L = dA.unsqueeze(-1) ** torch.arange(L, device=dA.device).float()  # [d_model, d_state, L]
        K = torch.einsum('dn,dnl->dl', C, dA_L * dB.unsqueeze(-1))  # [d_model, L]

        K = K.real  # 取实部
        return K


class S4Block(nn.Module):
    """完整的 S4 Block"""
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.ssm = S4Kernel(d_model, d_state)
        self.norm = nn.LayerNorm(d_model)
        self.proj_in = nn.Linear(d_model, d_model)
        self.proj_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        """x: [B, L, D]"""
        B, L, D = x.shape
        residual = x
        x = self.norm(x)
        x = self.proj_in(x)

        # 生成卷积核并做 1D 卷积
        K = self.ssm(L)  # [D, L]
        x = x.transpose(1, 2)  # [B, D, L]
        x = F.conv1d(x, K.unsqueeze(1))[:, :, :L]  # causal conv
        x = x.transpose(1, 2)  # [B, L, D]

        # D 残差
        x = x + self.ssm.D * residual

        x = self.proj_out(x)
        x = self.dropout(x)
        return x + residual


# 验证
s4 = S4Block(d_model=64, d_state=16).to(device)
x = torch.randn(2, 32, 64, device=device)
out = s4(x)
print(f'S4 Block: {x.shape} -> {out.shape}')
print(f'参数量: {sum(p.numel() for p in s4.parameters()):,}')

---
## 4. 选择性状态空间（Mamba 核心）

S4 的局限：B、C、$\Delta$ 是固定的（不依赖于输入），无法做选择性过滤。

Mamba 的核心创新：让 B、C、$\Delta$ **依赖于输入**。

$$B_t = \text{Linear}_B(x_t), \quad C_t = \text{Linear}_C(x_t), \quad \Delta_t = \text{softplus}(\text{Linear}_\Delta(x_t))$$

这使得模型可以**选择性地**记住或遗忘信息，类似注意力机制中的选择性关注。

由于参数依赖于输入，无法再用卷积模式，Mamba 使用**硬件感知的扫描算法**（parallel scan）。

In [ ]:
class SelectiveSSM(nn.Module):
    """Mamba 核心: 选择性状态空间模型
    
    这里使用朴素递推实现（教学用途），实际 Mamba 使用并行扫描优化。
    """
    def __init__(self, d_model, d_state=16, dt_min=0.001, dt_max=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # A 参数: 固定的对角矩阵（使用 HiPPO 初始化）
        A = torch.arange(1, d_state + 1).float().unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))

        # D: 直通连接
        self.D = nn.Parameter(torch.ones(d_model))

        # 输入投影（生成 B, C, dt）
        self.x_proj = nn.Linear(d_model, d_state * 2 + 1, bias=False)
        # d_state for B + d_state for C + 1 for dt

        # dt 投影
        self.dt_proj = nn.Linear(1, d_model, bias=True)
        # 初始化 dt 偏置使得初始 dt 在合理范围
        dt = torch.exp(
            torch.rand(d_model) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        )
        inv_dt = dt + torch.log(-torch.expm1(-dt))  # inverse of softplus
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_dt)

    def forward(self, x):
        """x: [B, L, D]"""
        B_batch, L, D = x.shape
        N = self.d_state

        # 固定的 A
        A = -torch.exp(self.A_log)  # [D, N] 保证负值（稳定）

        # 从输入生成 B, C, dt (选择性机制的核心)
        x_proj = self.x_proj(x)  # [B, L, 2*N + 1]
        B = x_proj[:, :, :N]       # [B, L, N]
        C = x_proj[:, :, N:2*N]    # [B, L, N]
        dt = x_proj[:, :, 2*N:2*N+1]  # [B, L, 1]

        dt = self.dt_proj(dt)  # [B, L, D]
        dt = F.softplus(dt)    # 保证 dt > 0

        # 离散化: A_bar = exp(dt * A), B_bar = dt * B (简化)
        dA = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))  # [B, L, D, N]
        dB = dt.unsqueeze(-1) * B.unsqueeze(2) * A.unsqueeze(0).unsqueeze(0)  # [B, L, D, N] 近似

        # 递推计算 (朴素实现)
        h = torch.zeros(B_batch, D, N, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            h = h * dA[:, t] + dB[:, t] * x[:, t].unsqueeze(-1)  # [B, D, N]
            y = torch.einsum('bdn,bn->bd', h, C[:, t])  # [B, D]
            ys.append(y)

        y = torch.stack(ys, dim=1)  # [B, L, D]
        y = y + self.D * x
        return y


# 验证
ssm = SelectiveSSM(d_model=32, d_state=8).to(device)
x = torch.randn(2, 16, 32, device=device)
out = ssm(x)
print(f'Selective SSM: {x.shape} -> {out.shape}')

# 对比固定 vs 选择性
print('''\n=== 选择性机制 vs 固定参数 ===
S4:   B, C, Δ 是固定参数 → 对所有输入一视同仁
Mamba: B, C, Δ 依赖于输入 → 可以选择性记住/遗忘
      类比: 注意力机制中不同 token 获得不同的注意力权重''')

---
## 5. Mamba Block 完整实现

In [ ]:
class MambaBlock(nn.Module):
    """完整的 Mamba Block
    
    结构: Input → [Linear → Conv1D → SiLU → SSM → Linear] + Input
    """
    def __init__(self, d_model, d_state=16, d_conv=4, expand_factor=2):
        super().__init__()
        self.d_model = d_model
        self.d_inner = d_model * expand_factor

        self.norm = nn.LayerNorm(d_model)
        self.proj_in = nn.Linear(d_model, self.d_inner * 2, bias=False)

        # Causal Conv1D
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner,
            kernel_size=d_conv, padding=d_conv - 1,
            groups=self.d_inner
        )

        # SSM
        self.ssm = SelectiveSSM(self.d_inner, d_state)

        # 输出投影
        self.proj_out = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        """x: [B, L, D]"""
        B, L, D = x.shape
        residual = x

        x = self.norm(x)
        xz = self.proj_in(x)  # [B, L, 2*d_inner]
        x, z = xz.chunk(2, dim=-1)  # 各 [B, L, d_inner]

        # Causal Conv1D
        x = x.transpose(1, 2)  # [B, d_inner, L]
        x = self.conv1d(x)[:, :, :L]  # 截断保证因果性
        x = x.transpose(1, 2)  # [B, L, d_inner]

        # SiLU 激活
        x = F.silu(x)

        # SSM
        x = self.ssm(x)

        # 门控
        x = x * F.silu(z)

        # 输出
        x = self.proj_out(x)
        return x + residual


class MambaModel(nn.Module):
    """Mamba 语言模型"""
    def __init__(self, vocab_size, d_model=256, n_layers=4, d_state=16, d_conv=4, expand_factor=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            MambaBlock(d_model, d_state, d_conv, expand_factor)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return self.head(x)


# 验证
model = MambaModel(vocab_size=100, d_model=64, n_layers=2, d_state=8).to(device)
x = torch.randint(0, 100, (2, 32), device=device)
out = model(x)
total_params = sum(p.numel() for p in model.parameters())
print(f'Mamba Model: {x.shape} -> {out.shape}')
print(f'参数量: {total_params:,}')

---
## 6. 实战：Mamba 文本生成模型

使用与 Notebook 01 相同的数据集，对比 Mamba 和 Transformer 的学习效果。

In [ ]:
# 准备数据（同 Notebook 01）
text = """The transformer architecture has revolutionized natural language processing. 
Self-attention allows the model to weigh the importance of different words in a sequence. 
The key innovation is the ability to process all positions in parallel, unlike recurrent networks. 
Multi-head attention enables the model to attend to information from different representation subspaces. 
Position encoding provides the model with sequence order information. 
The encoder-decoder structure is used for sequence-to-sequence tasks like translation. 
Decoder-only models like GPT are widely used for text generation. 
Encoder-only models like BERT excel at understanding tasks such as classification. 
State space models offer a compelling alternative to transformers for long sequence modeling. 
Mamba introduces selective state spaces that can choose what information to remember or forget."""

chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long).to(device)

print(f'词表大小: {vocab_size}, 文本长度: {len(data)}')

In [ ]:
# 训练 Mamba
mamba = MambaModel(vocab_size, d_model=64, n_layers=2, d_state=8, expand_factor=2).to(device)
optimizer = torch.optim.AdamW(mamba.parameters(), lr=3e-4)

block_size = 32
batch_size = 8
losses_mamba = []

mamba.train()
for step in range(200):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    
    logits = mamba(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(mamba.parameters(), 1.0)
    optimizer.step()
    losses_mamba.append(loss.item())
    
    if step % 50 == 0:
        print(f'Step {step:4d} | Loss: {loss.item():.4f}')

plt.figure(figsize=(8, 3))
plt.plot(losses_mamba)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Mamba Training Loss')
plt.show()

In [ ]:
# Mamba 自回归生成
def generate_mamba(model, idx, max_new_tokens, temperature=0.8):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -64:]
        logits = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, idx_next], dim=1)
    return idx

context = torch.tensor([[char_to_idx['T']]], device=device)
generated = generate_mamba(mamba, context, max_new_tokens=150, temperature=0.7)
text_gen = ''.join([idx_to_char[i] for i in generated[0].tolist()])
print('=== Mamba 生成结果 ===')
print(text_gen)

---
## 复杂度对比

| 模型 | 训练复杂度 | 推理复杂度 | 长序列能力 | 代表 |
|------|-----------|-----------|-----------|------|
| Transformer | $O(n^2 d)$ | $O(n^2 d)$ | 受限于 $n^2$ | GPT, BERT |
| S4 (卷积) | $O(n \log n \cdot d)$ | $O(n d)$ | 优秀 | S4, S4ND |
| Mamba (SSM) | $O(n d)$ | $O(n d)$ | 优秀 | Mamba |
| RWKV (线性RNN) | $O(n d)$ | $O(d)$ | 优秀 | RWKV |

### Mamba vs Transformer 的核心区别

1. **注意力 vs 状态**: Transformer 用注意力矩阵显式建模所有 token 关系；Mamba 用压缩的隐状态
2. **O(n^2) vs O(n)**: Mamba 在训练和推理上都是线性复杂度
3. **选择性**: Mamba 通过输入依赖的 B/C/Δ 实现类似注意力的选择性
4. **推理加速**: Mamba 不需要 KV-Cache，推理时内存恒定

### Mamba 的局限
- 在需要精确回溯的任务上可能不如 Transformer
- 生态和工具链不如 Transformer 成熟
- Mamba-2 进一步优化了硬件效率（结构化注意力与 SSM 的统一）